In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   SARCASM DETECTOR  —  FILE 1: TRAIN & SAVE                                  ║
# ║                                                                              ║
# ║   Run this in Google Colab (T4 GPU) to train and push to HuggingFace Hub.    ║
# ║                                                                              ║
# ║   STEPS:                                                                     ║
# ║     1. Runtime → Change runtime type → T4 GPU                                ║
# ║     2. Add your HF write token to Colab Secrets as  HF_TOKEN                 ║
# ║     3. Make sure danofer-sarcasm.zip is in your Google Drive root            ║
# ║     4. Paste entire file into one cell → Run                                 ║
# ║     5. After training, model is pushed to HuggingFace Hub automatically      ║
# ║                                                                              ║
# ║   Output: YOUR_HF_USERNAME/sarcasm-roberta  on HuggingFace Hub               ║
# ║   Est. training time: ~30 min on T4 GPU                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── CUDA safety — must be before torch import ─────────────────────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  INSTALL
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["transformers>=4.40", "datasets", "scikit-learn",
            "accelerate", "huggingface_hub"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

# ═══════════════════════════════════════════════════════════════════════════════
# 2.  IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import warnings, re, json
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from torch.nn import CrossEntropyLoss
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)
from datasets import load_dataset as hf_load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}  |  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════════════════════════
# 3.  CONFIG  ← edit these
# ═══════════════════════════════════════════════════════════════════════════════
HF_REPO_NAME = "sarcasm-roberta"
DRIVE_ZIP    = "/content/drive/MyDrive/danofer-sarcasm.zip"
REDDIT_NROWS = 90_000                     # rows to load from Reddit SARC
SAVE_DIR     = "./roberta_sarcasm_model"
DRIVE_SAVE   = "/content/drive/MyDrive/totally-not-sarcastic"

CFG = dict(
    model_name   = "roberta-base",
    max_len      = 128,
    epochs       = 3,
    batch_gpu    = 32,
    batch_cpu    = 8,
    lr           = 2e-5,
    weight_decay = 0.01,
    warmup_ratio = 0.06,
    label_smooth = 0.05,
    grad_clip    = 1.0,
    seed         = 42,
    val_size     = 0.10,
    max_cpu_rows = 4_000,
)
BATCH = CFG["batch_gpu"] if device.type == "cuda" else CFG["batch_cpu"]

# Precision: RoBERTa → fp16, DeBERTa-v3 → bf16
IS_DEBERTA = "deberta" in CFG["model_name"].lower()
USE_FP16   = (device.type == "cuda") and not IS_DEBERTA
USE_BF16   = (device.type == "cuda") and IS_DEBERTA and torch.cuda.is_bf16_supported()
_prec = "fp16" if USE_FP16 else ("bf16" if USE_BF16 else "fp32")
print(f"⚙️  Model={CFG['model_name']}  batch={BATCH}  epochs={CFG['epochs']}  prec={_prec}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4.  TEXT CLEANING
# ═══════════════════════════════════════════════════════════════════════════════
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_MD_RE  = re.compile(r"[*_~`>|#]")
_WS_RE  = re.compile(r"\s{2,}")

def clean(text: str) -> str:
    if not isinstance(text, str): return ""
    text = _URL_RE.sub("", text)
    text = _MD_RE.sub("", text)
    text = text.replace("\n", " ").replace("\r", " ")
    return _WS_RE.sub(" ", text).strip()[:512]

# ═══════════════════════════════════════════════════════════════════════════════
# 5.  LOAD DATASETS
# ═══════════════════════════════════════════════════════════════════════════════
frames = []

# ── 5a. Reddit SARC from Google Drive zip ────────────────────────────────────
print("\n📥  Reddit SARC from Google Drive…")
try:
    import zipfile, glob
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    if not os.path.exists(DRIVE_ZIP):
        raise FileNotFoundError(
            f"Zip not found at {DRIVE_ZIP}\n"
            "     → Put danofer-sarcasm.zip in the ROOT of My Drive.")
    os.makedirs("/tmp/sarcasm_kaggle", exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        zf.extractall("/tmp/sarcasm_kaggle")
    csvs = glob.glob("/tmp/sarcasm_kaggle/**/*.csv", recursive=True)
    if not csvs:
        raise FileNotFoundError("No CSV inside zip.")
    kaggle_csv = next((p for p in csvs if "train-balanced" in p.lower()), csvs[0])
    df_reddit  = pd.read_csv(kaggle_csv, nrows=REDDIT_NROWS)
    col_map = {}
    for c in df_reddit.columns:
        lc = c.lower()
        if "parent" in lc:                                         col_map[c] = "context"
        elif "comment" in lc or "response" in lc or "text" in lc: col_map[c] = "response"
        elif "label" in lc or "sarc" in lc:                       col_map[c] = "label"
    df_reddit.rename(columns=col_map, inplace=True)
    if "context"  not in df_reddit.columns: df_reddit["context"] = ""
    df_reddit = df_reddit[["label","response","context"]].dropna(subset=["response"])
    df_reddit["label"]    = df_reddit["label"].astype(int)
    df_reddit["response"] = df_reddit["response"].apply(clean)
    df_reddit["context"]  = df_reddit["context"].apply(clean)
    frames.append(df_reddit)
    has_ctx = (df_reddit["context"] != "").sum()
    print(f"  ✅ Reddit SARC  →  {len(df_reddit):,} rows  |  context pairs: {has_ctx:,}")
except Exception as e:
    print(f"  ⚠️  Reddit SARC failed: {e}")

# ── 5b. TweetEval Irony ───────────────────────────────────────────────────────
print("\n📥  TweetEval Irony…")
try:
    ds = hf_load("tweet_eval", "irony", split="train+test", trust_remote_code=False)
    df = pd.DataFrame({"response": [clean(t) for t in ds["text"]],
                       "label": ds["label"], "context": ""})
    frames.append(df)
    print(f"  ✅ TweetEval  →  {len(df):,} rows")
except Exception as e:
    print(f"  ⚠️  TweetEval: {e}")

# ── 5c. News Headlines ────────────────────────────────────────────────────────
print("\n📥  Sarcasm Headlines…")
for owner, name in [("raquiba","Sarcasm_News_Headline"), ("jakehobbs","sarcasm")]:
    try:
        ds   = hf_load(f"{owner}/{name}", split="train", trust_remote_code=False)
        tc   = next(c for c in ds.column_names if "headline" in c.lower() or "text" in c.lower())
        lc   = next(c for c in ds.column_names if "sarc" in c.lower() or "label" in c.lower())
        df   = pd.DataFrame({"response": [clean(t) for t in ds[tc]],
                              "label": [int(l) for l in ds[lc]], "context": ""})
        frames.append(df)
        print(f"  ✅ {owner}/{name}  →  {len(df):,} rows")
        break
    except Exception as e:
        print(f"  ⚠️  {owner}/{name}: {e}")

# ── 5d. Merge + balance ───────────────────────────────────────────────────────
if not frames:
    raise RuntimeError("❌ No datasets loaded. Check Drive zip and internet access.")

combined = pd.concat(frames, ignore_index=True)
combined = combined.dropna(subset=["response"])
combined = combined[combined["response"].str.strip() != ""]
combined["context"]  = combined["context"].fillna("").astype(str)
combined["response"] = combined["response"].astype(str)
combined["label"]    = combined["label"].astype(int)

if device.type == "cpu":
    MAX = CFG["max_cpu_rows"]
    combined = (combined.groupby("label", group_keys=False)
                        .apply(lambda g: g.sample(min(MAX, len(g)), random_state=42))
                        .reset_index(drop=True))
    print(f"⚠️  CPU mode — trimmed to {len(combined):,} samples")

min_cls  = combined["label"].value_counts().min()
combined = (combined.groupby("label", group_keys=False)
                    .apply(lambda g: g.sample(min(min_cls, len(g)), random_state=42))
                    .reset_index(drop=True)
                    .sample(frac=1, random_state=42).reset_index(drop=True))

SOURCES_USED = [f"{len(f):,} rows" for f in frames]
print(f"\n📊 Dataset: {len(combined):,} balanced samples")
print(f"   Sarcastic  : {combined['label'].sum():,}")
print(f"   Normal     : {(combined['label']==0).sum():,}")
print(f"   Has context: {(combined['context']!='').sum():,}")

# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TOKENIZER
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n🔤  Loading tokenizer ({CFG['model_name']})…")
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
print(f"✅  vocab_size={tokenizer.vocab_size:,}")

def encode_pairs(ctxs, ress):
    has_any = any(c and c.strip() for c in ctxs)
    if has_any:
        text_a = [c.strip() if (c and c.strip()) else "" for c in ctxs]
        return tokenizer(text=text_a, text_pair=list(ress),
                         truncation="longest_first", padding=True,
                         max_length=CFG["max_len"], return_tensors=None)
    return tokenizer(text=list(ress), truncation=True, padding=True,
                     max_length=CFG["max_len"], return_tensors=None)

(ctx_tr, ctx_val, res_tr, res_val, y_tr, y_val) = train_test_split(
    combined["context"].tolist(), combined["response"].tolist(),
    combined["label"].tolist(),
    test_size=CFG["val_size"], random_state=CFG["seed"],
    stratify=combined["label"].tolist())

print(f"   Train: {len(y_tr):,}  |  Val: {len(y_val):,}")
print("   Tokenizing…")
train_enc = encode_pairs(ctx_tr, res_tr)
val_enc   = encode_pairs(ctx_val, res_val)
print("✅  Tokenization done!")

# ═══════════════════════════════════════════════════════════════════════════════
# 7.  DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class SarcasmDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc    = enc
        self.labels = labels
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_ds = SarcasmDataset(train_enc, y_tr)
val_ds   = SarcasmDataset(val_enc,   y_val)

# ═══════════════════════════════════════════════════════════════════════════════
# 8.  CUSTOM TRAINER — label smoothing
# ═══════════════════════════════════════════════════════════════════════════════
class SmoothedTrainer(Trainer):
    def __init__(self, *args, label_smoothing=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self._ls = label_smoothing
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = CrossEntropyLoss(label_smoothing=self._ls)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ═══════════════════════════════════════════════════════════════════════════════
# 9.  MODEL
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n🧠  Loading {CFG['model_name']}…")
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"], num_labels=2, ignore_mismatched_sizes=True)
model.to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅  {n_params:.0f}M parameters")

# ═══════════════════════════════════════════════════════════════════════════════
# 10. TRAIN
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(pred):
    lbls  = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(lbls, preds, average="binary")
    return {"accuracy": round(accuracy_score(lbls, preds), 4),
            "f1": round(f1, 4), "precision": round(p, 4), "recall": round(r, 4)}

TOTAL_STEPS  = (len(train_ds) // BATCH) * CFG["epochs"]
WARMUP_STEPS = int(TOTAL_STEPS * CFG["warmup_ratio"])

args = TrainingArguments(
    output_dir                  = "./ckpt",
    num_train_epochs            = CFG["epochs"],
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    logging_strategy            = "epoch",
    learning_rate               = CFG["lr"],
    weight_decay                = CFG["weight_decay"],
    warmup_steps                = WARMUP_STEPS,
    lr_scheduler_type           = "cosine",
    max_grad_norm               = CFG["grad_clip"],
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    report_to                   = "none",
    seed                        = CFG["seed"],
)

trainer = SmoothedTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    label_smoothing = CFG["label_smooth"],
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"\n🚀  Training  |  Epochs: {CFG['epochs']}  |  "
      f"Batch: {BATCH}  |  LR: {CFG['lr']}  |  Warmup: {WARMUP_STEPS}")
trainer.train()

# ═══════════════════════════════════════════════════════════════════════════════
# 11. EVALUATE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n📊  Final Evaluation…")
eval_res  = trainer.evaluate()
val_preds = np.argmax(trainer.predict(val_ds).predictions, axis=1)
cm        = confusion_matrix(y_val, val_preds)

print(f"\n{'─'*44}")
print(f"  Accuracy  : {eval_res['eval_accuracy']*100:.2f}%")
print(f"  F1 Score  : {eval_res['eval_f1']*100:.2f}%")
print(f"  Precision : {eval_res['eval_precision']*100:.2f}%")
print(f"  Recall    : {eval_res['eval_recall']*100:.2f}%")
print(f"{'─'*44}")
print(classification_report(y_val, val_preds, target_names=["Not Sarcastic","Sarcastic"]))

# ═══════════════════════════════════════════════════════════════════════════════
# 12. SAVE LOCALLY + TO DRIVE
# ═══════════════════════════════════════════════════════════════════════════════
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\n💾  Saved locally → {SAVE_DIR}")

# Save training metadata so the dashboard can read it
import json as _json
meta = {
    "model_name":    CFG["model_name"],
    "accuracy":      round(eval_res["eval_accuracy"], 4),
    "f1":            round(eval_res["eval_f1"], 4),
    "precision":     round(eval_res["eval_precision"], 4),
    "recall":        round(eval_res["eval_recall"], 4),
    "epochs":        CFG["epochs"],
    "lr":            CFG["lr"],
    "batch":         BATCH,
    "max_len":       CFG["max_len"],
    "label_smooth":  CFG["label_smooth"],
    "train_samples": len(y_tr),
    "val_samples":   len(y_val),
    "total_samples": len(combined),
    "sources":       SOURCES_USED,
    "has_context":   int((combined["context"] != "").sum()),
    "n_params_M":    round(n_params, 1),
    "confusion_matrix": cm.tolist(),
    "precision_str": _prec,
}
with open(f"{SAVE_DIR}/training_meta.json", "w") as f:
    _json.dump(meta, f, indent=2)
print("📋  training_meta.json saved")

try:
    import shutil
    shutil.copytree(SAVE_DIR, DRIVE_SAVE, dirs_exist_ok=True)
    print(f"💾  Also saved to Google Drive → {DRIVE_SAVE}")
except Exception as e:
    print(f"⚠️  Drive save failed: {e}")

# ═══════════════════════════════════════════════════════════════════════════════
# 13. PUSH TO HUGGINGFACE HUB
# ═══════════════════════════════════════════════════════════════════════════════
print("\n🤗  Pushing to HuggingFace Hub…")
try:
    from google.colab import userdata
    from huggingface_hub import HfApi, login

    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)

    api = HfApi()
    username = api.whoami()["name"]
    repo_id  = f"{username}/{HF_REPO_NAME}"

    # Create repo if needed
    api.create_repo(repo_id=repo_id, exist_ok=True, private=False)

    # Upload model files
    api.upload_folder(folder_path=SAVE_DIR, repo_id=repo_id, repo_type="model")

    print(f"✅  Model live at: https://huggingface.co/{repo_id}")
    print(f"   Use this repo_id in File 2:  MODEL_REPO = \"{repo_id}\"")

except Exception as e:
    print(f"⚠️  HF push failed: {e}")
    print("   → Make sure HF_TOKEN is in Colab Secrets with write access")
    print(f"   → Model is still saved locally at {SAVE_DIR} and on Drive")

print("\n✅  Training complete!")

✅ Device: cuda
✅ GPU: Tesla T4  |  15.6 GB
⚙️  Model=roberta-base  batch=32  epochs=3  prec=fp16

📥  Reddit SARC from Google Drive…
  ✅ Reddit SARC  →  89,997 rows  |  context pairs: 89,993

📥  TweetEval Irony…
  ✅ TweetEval  →  3,646 rows

📥  Sarcasm Headlines…


Repo card metadata block was not found. Setting CardData to empty.


  ✅ raquiba/Sarcasm_News_Headline  →  28,619 rows

📊 Dataset: 107,058 balanced samples
   Sarcastic  : 53,529
   Normal     : 53,529
   Has context: 78,411

🔤  Loading tokenizer (roberta-base)…
✅  vocab_size=50,265
   Train: 96,352  |  Val: 10,706
   Tokenizing…
✅  Tokenization done!

🧠  Loading roberta-base…


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


💾  Saved locally → ./roberta_sarcasm_model
📋  training_meta.json saved
💾  Also saved to Google Drive → /content/drive/MyDrive/totally-not-sarcastic

🤗  Pushing to HuggingFace Hub…


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...m_model/model.safetensors:   0%|          |  551kB /  499MB            

✅  Model live at: https://huggingface.co/AK-Rahul/sarcasm-roberta
   Use this repo_id in File 2:  MODEL_REPO = "AK-Rahul/sarcasm-roberta"

✅  Training complete!
